# Analisis Kesesuaian Star Rating dan Sentimen Ulasan
## Menggunakan Metode Multinomial Naive Bayes

Proyek ini bertujuan untuk membangun model klasifikasi sentimen otomatis untuk ulasan aplikasi Google Play Store dan membandingkan hasilnya dengan rating bintang yang diberikan pengguna.

### 1. Persiapan Lingkungan dan Import Library
Tahap ini mencakup pemuatan pustaka yang diperlukan untuk manipulasi data, pemrosesan teks, dan pembangunan model machine learning.

In [ ]:
from imblearn.over_sampling import RandomOverSampler
from sklearn.model_selection import StratifiedKFold, cross_val_score
import pandas as pd
import numpy as np
import re
import kagglehub
from kagglehub import KaggleDatasetAdapter

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')from sklearn.metrics import balanced_accuracy_score, f1_score


### 2. Pemuatan Dataset dari Kaggle
Menggunakan `kagglehub` untuk menarik data terbaru secara otomatis dari repositori publik Kaggle.

In [ ]:
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "hassaanmustafavi/google-play-app-reviews-dataset",
  "GooglePlay_App_Data.csv",
)

# Lihat data awal

In [ ]:
df.head()

### 3. Pembersihan dan Transformasi Data
Melakukan pembersihan data mentah, menghapus nilai null, dan memetakan kolom ke format yang lebih konsisten.

In [ ]:
df = df.dropna(subset=['review_description'])


# Rename biar lebih enak

In [ ]:
df = df.rename(columns={
    "review_description": "review",
    "rating": "star_rating"
})

df.head()

### 4. Text Preprocessing
Langkah krusial untuk mengubah teks mentah menjadi format yang bersih melalui:
- Lowercasing
- Removal of special characters
- Stopwords removal

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = text.split()
    text = [word for word in text if word not in stop_words]
    return " ".join(text)

df['clean_review'] = df['review'].apply(clean_text)

df[['review', 'clean_review']].head()

### 5. Pelabelan Sentimen Berdasarkan Star Rating
Mengonversi nilai numerik rating (1-5) menjadi label kategorikal (Positive, Neutral, Negative) sebagai *ground truth* untuk evaluasi.

In [ ]:
df['sentiment'] = df['star_rating'].apply(lambda x:
    'positive' if x >= 4 else
    'neutral' if x == 3 else
    'negative'
)

df['sentiment'].value_counts()

### 6. Ekstraksi Fitur dengan TF-IDF
Mengubah teks ulasan menjadi representasi vektor numerik menggunakan metode *Term Frequency-Inverse Document Frequency*.

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df['clean_review'])
y = df['sentiment']

# =========================================
# 7. SPLIT DATA
# =========================================

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### 7.1. Menangani Ketidakseimbangan Data (Oversampling)
Karena data sangat didominasi oleh ulasan positif, kita menggunakan `RandomOverSampler` agar model dapat mempelajari pola dari kelas minoritas (negatif dan netral).

In [ ]:
ros = RandomOverSampler(random_state=42)
X_train_res, y_train_res = ros.fit_resample(X_train, y_train)

print("Distribusi label sebelum oversampling:", y_train.value_counts())
print("Distribusi label setelah oversampling:", pd.Series(y_train_res).value_counts())

### 7. Pelatihan Model Multinomial Naive Bayes
Naive Bayes dipilih karena efisiensinya dalam menangani fitur teks yang memiliki dimensi tinggi.

In [ ]:
model = MultinomialNB()
model.fit(X_train_res, y_train_res)

# =========================================
# 9. PREDIKSI
# =========================================

### 7.2. Cross-Validation
Melakukan validasi silang untuk memastikan model stabil di berbagai partisi data.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=skf, scoring='f1_weighted')

print(f"F1-Score (Weighted) Cross-Val: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

In [ ]:
y_pred = model.predict(X_test)


### 8. Evaluasi Kinerja Model
Menganalisis performa model menggunakan metrik standar seperti Accuracy, Classification Report, dan Confusion Matrix.

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("F1-Score (Macro):", f1_score(y_test, y_pred, average='macro'))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

### 9. Analisis Kesesuaian (Simulasi Konsep Skripsi)
Membandingkan hasil prediksi model dengan label asli untuk melihat tingkat kecocokan antara persepsi model dan rating pengguna.

In [ ]:
comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

comparison['match'] = comparison['Actual'] == comparison['Predicted']

# Hitung kesesuaian

In [ ]:
accuracy_match = comparison['match'].mean()

print("\nTingkat Kesesuaian (Simulasi):", accuracy_match)

comparison.head()

# =========================================
# 12. SAVE MODEL (OPSIONAL)
# =========================================

In [ ]:
import pickle

pickle.dump(model, open("model_naive_bayes.pkl", "wb"))
pickle.dump(vectorizer, open("tfidf_vectorizer.pkl", "wb"))